In [2]:
# 1. Imports
import pandas as pd
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import optuna

In [ ]:
# 2. Cargar y preprocesar los datos (igual que antes)
data = pd.read_csv('insurance.csv')

encoders = {}
for col in ['sex', 'smoker', 'region']:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

X = data.drop(columns=['charges'])
y = data['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 3. Definir la función objetivo para Optuna
def objective(trial):
    # Optuna va a probar diferentes valores dentro de estos rangos
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    
    # Crear el modelo con los hiperparámetros sugeridos por Optuna en este intento
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    
    # Evaluamos el modelo usando validación cruzada (negativo del error medio absoluto o MSE)
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_mean_squared_error')
    mean_score = scores.mean()
    
    # Optuna por defecto busca minimizar, así que devolvemos el error (convertido a positivo) o maximizamos la métrica
    return -mean_score

In [ ]:
# 4. Crear el estudio de Optuna y ejecutar la búsqueda (por ejemplo, 20 pruebas/intentos)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

In [ ]:
# 5. Ver cuáles fueron los mejores hiperparámetros encontrados
print("Mejores hiperparámetros encontrados:", study.best_params)

In [ ]:
# 6. Entrenar el modelo final con esos mejores parámetros
best_params = study.best_params
final_model = RandomForestRegressor(**best_params, random_state=42)
final_model.fit(X_train, y_train)

# 6. Guardar el modelo optimizado y los encoders
with open('modelo_seguros.pkl', 'wb') as f:
    pickle.dump(final_model, f)

with open('encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print("¡Modelo optimizado con Optuna y guardado con éxito!")

In [ ]:
# 7. Guardar el modelo optimizado y los encoders
with open('modelo_seguros.pkl', 'wb') as f:
    pickle.dump(final_model, f)

with open('encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print("¡Modelo optimizado con Optuna y guardado con éxito!")